# 04 – Supplementary Fig. S1: Nationwide Similarity Maps by Structural Type

Regenerates the 35 per-park nationwide similarity maps (Fig. S1) using the corrected
elevation/slope DEM (`fix/` pipeline), grouped by each park's (re-)computed structural
type — Type I / II / III / Island — as reported in Supplementary Table S2.

Each map plots H3 resolution-9 cells whose park-specific similarity score is at or above
that park's production threshold (Supplementary Table S1), binned into 5 equal-interval
classes and colored with the Turbo colormap (matching the QGIS palette used for the
original Fig. S1).

**Prerequisites** (produced by `02_modeling.ipynb` / `03_analysis_figures.ipynb` on the
`fix/` dataset):
- `fix/data/interim/h3_jpn_res9_source_imputed.parquet`
- `fix/paper_outputs/nationwide_similarity_all_parks.parquet`
- `fix/paper_outputs/SupplementaryTableS1_model_configuration.csv`
- `fix/paper_outputs/SupplementaryTableS2_similarity_surface.csv`

**Coordinate reference system.** All coordinates are handled as geographic WGS 84
(EPSG:4326) — the same CRS as the project-level CRS in
`qgis/Project_Files/Sup_Fig1_result_for_35_parks.qgz`. EPSG:4326 is an *unprojected*
(angular) CRS, so a scale bar cannot simply be measured in plot units. Instead, the scale
bar's length in degrees is computed geodesically on the WGS84 ellipsoid (`pyproj.Geod`) for
the map's central latitude, so the labelled distance is metrically accurate even though the
plot itself is unprojected — this mirrors what QGIS does internally when a scale bar is
added to a print layout under a geographic-CRS project.

**North arrow / scale bar convention.** Because all 35 maps share the identical extent and
orientation, the north arrow and scale bar are drawn once (see the last section) rather than
on every individual tile, to avoid repeating identical information 35 times. Each individual
map instead carries its own legend, since the similarity-score binning and threshold are
park-specific.

## Imports and configuration

In [ ]:
import json
from pathlib import Path

import duckdb
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from pyproj import Geod
from shapely.geometry import Point

DATA_DIR   = Path("../fix/data/interim")   # h3_jpn_res9_source_imputed.parquet

PAPER_DIR  = Path("../fix/paper_outputs")  # tables/figures produced by 03_analysis_figures.ipynb

OUT_DIR    = PAPER_DIR / "FigS1_check_new_threshold"  # output: 35 PNGs, grouped by type


SIMILARITY_PARQUET = PAPER_DIR / "nationwide_similarity_all_parks.parquet"
SOURCE_PARQUET      = DATA_DIR / "h3_jpn_res9_source_imputed.parquet"
TABLE_S1_CSV        = PAPER_DIR / "SupplementaryTableS1_model_configuration.csv"
TABLE_S2_CSV        = PAPER_DIR / "SupplementaryTableS2_similarity_surface.csv"

CRS_WGS84 = "EPSG:4326"

for sub in ["TypeI", "TypeII", "TypeIII", "Island"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

## Park configuration: display names, production thresholds, structural type

`PARK_DISPLAY` mirrors the dict used in `03_analysis_figures.ipynb` (English slug → manuscript
display name). Thresholds and structural-type labels are read directly from Supplementary
Tables S1 and S2, rather than recomputed here, so this notebook always reflects whatever the
`03_analysis_figures.ipynb` run most recently produced.

In [ ]:
PARK_DISPLAY = {

    "rishiri":      "1. Rishiri-Rebun-Sarobetsu",

    "shiretoko":    "2. Shiretoko",

    "akan":         "3. Akan-Mashu",

    "kushiro":      "4. Kushiroshitsugen",

    "taisetsu":     "5. Daisetsuzan",

    "hidaka":       "6. Hidakasanmyaku-Erimo-Tokachi",

    "shikotsu":     "7. Shikotsu-Toya",

    "towada":       "8. Towada-Hachimantai",

    "sanriku":      "9. Sanriku Fukko",

    "bandai":       "10. Bandai-Asahi",

    "nikko":        "11. Nikko",

    "oze":          "12. Oze",

    "jyoshinetsu":  "13. Joshin'etsukogen",

    "myoko":        "14. Myoko-Togakushi renzan",

    "chichibu":     "15. Chichibu-Tama-Kai",

    "ogasawara":    "16. Ogasawara",

    "fuji":         "17. Fuji-Hakone-Izu",

    "chubusangaku": "18. Chūbu-Sangaku",

    "hakusan":      "19. Hakusan",

    "minamialps":   "20. Minami Alps",

    "ise":          "21. Ise-Shima",

    "yoshino":      "22. Yoshino-Kumano",

    "sanin":        "23. San'inkaigan",

    "setonaikai":   "24. Setonaikai",

    "daisen":       "25. Daisen-Oki",

    "ashizuri":     "26. Ashizuri-Uwakai",

    "saikai":       "27. Saikai",

    "unzen":        "28. Unzen-Amakusa",

    "aso":          "29. Aso-Kuju",

    "kirishima":    "30. Kirishima-Kinkowan",

    "yakushima":    "31. Yakushima (Island)",

    "amami":        "32. Amamigunto",

    "yambaru":      "33. Yambaru",

    "kerama":       "34. Keramashoto",

    "iriomote":     "35. Iriomote-Ishigaki",

}

s1 = pd.read_csv(TABLE_S1_CSV).set_index("Park")
s2 = pd.read_csv(TABLE_S2_CSV).set_index("Park")

park_config = {}
for slug, disp in PARK_DISPLAY.items():
    park_config[slug] = {
        "display":   disp,
        "threshold": float(s1.loc[disp, "Production threshold"]),
        # "Type I" / "Type II" / "Type III" / "Island" -> "TypeI" / ... (folder-safe name)
        "type":      s2.loc[disp, "Structural type"].replace(" ", ""),
    }

pd.DataFrame(park_config).T

## Background context points

A faint nationwide sketch of Japan's outline, built once from the full H3 grid and reused
as a shared background layer across all 35 maps (avoids reloading ~4M rows 35 times).
Every 15th cell is kept, which is dense enough to trace the coastline while keeping
plotting fast. CRS is EPSG:4326 since H3 cell centroids are native WGS84 lat/lon.

In [ ]:
bg = pd.read_parquet(SOURCE_PARQUET, columns=["h3_9"])
bg_cells = bg["h3_9"].values[::15]  # ~270k points
bg_latlng = np.array([h3.cell_to_latlng(c) for c in bg_cells])

bg_gdf = gpd.GeoDataFrame(
    geometry=[Point(lng, lat) for lat, lng in bg_latlng],
    crs=CRS_WGS84,
)
bg_lat = bg_gdf.geometry.y.values
bg_lng = bg_gdf.geometry.x.values
print(f"Background points: {len(bg_lat):,}  (CRS: {bg_gdf.crs})")

## Efficient per-park extraction

`nationwide_similarity_all_parks.parquet` is a long-format table (~4.05M H3 cells × 35
parks ≈ 142M rows, several GB). It is written **sorted by park** (contiguous row groups
per park), so filtering on `park = ...` lets DuckDB skip almost all row groups instead of
scanning the full file — each park's extraction takes ~1–3 seconds rather than minutes.
Filtering on `similarity_score >= threshold` in the same query keeps only the (much
smaller) set of cells that will actually be plotted.

In [ ]:
def get_park_similarity(slug: str, threshold: float, con: duckdb.DuckDBPyConnection) -> gpd.GeoDataFrame:
    df = con.execute(
        f"""
        SELECT h3_9, similarity_score
        FROM '{SIMILARITY_PARQUET.as_posix()}'
        WHERE park = ? AND similarity_score >= ?
        """,
        [slug, threshold],
    ).fetchdf()

    latlng = np.array([h3.cell_to_latlng(c) for c in df["h3_9"].values])
    gdf = gpd.GeoDataFrame(
        df.assign(lat=latlng[:, 0], lng=latlng[:, 1]),
        geometry=gpd.points_from_xy(latlng[:, 1], latlng[:, 0]),
        crs=CRS_WGS84,
    )
    return gdf

## Geodesic scale bar and north arrow

Because the map is plotted directly in EPSG:4326 degrees (no projection), 1° of longitude
does **not** correspond to a fixed ground distance — it shrinks toward the poles. A scale
bar drawn as a fixed number of *degrees* would therefore be wrong. Instead,
`geodesic_scalebar_deg` asks pyproj/WGS84 "how many degrees of longitude, at this map's
central latitude, correspond to `km` kilometres on the ground?" and returns that span, so the
drawn bar is a metrically correct ruler for that latitude. This is the same ellipsoidal
calculation QGIS performs internally for scale bars in geographic-CRS print layouts.

As agreed, this is drawn **once** (not on every one of the 35 tiles) since every map shares
the same extent, projection and orientation — see the final section.

In [ ]:
_GEOD = Geod(ellps="WGS84")

def geodesic_scalebar_deg(center_lat: float, center_lng: float, km: float) -> float:
    """Longitude span (degrees) equivalent to `km` kilometres at `center_lat`,
    computed on the WGS84 ellipsoid."""
    lon2, _, _ = _GEOD.fwd(center_lng, center_lat, 90, km * 1000)
    return lon2 - center_lng

def draw_scalebar_and_north_arrow(ax, bg_lat, bg_lng, km=200):
    center_lat = np.mean(bg_lat)
    center_lng = np.mean(bg_lng)
    span_deg = geodesic_scalebar_deg(center_lat, center_lng, km)

    x0 = bg_lng.max() - 0.5 - span_deg
    y0 = bg_lat.min() + 0.3
    ax.plot([x0, x0 + span_deg], [y0, y0], color="black", lw=2, solid_capstyle="butt")
    ax.plot([x0, x0], [y0 - 0.05, y0 + 0.05], color="black", lw=2)
    ax.plot([x0 + span_deg, x0 + span_deg], [y0 - 0.05, y0 + 0.05], color="black", lw=2)
    ax.text(x0 + span_deg / 2, y0 + 0.12, f"{km} km", ha="center", fontsize=11)

    # North is straight up in an unprojected (PlateCarree-like) lat/lon plot.
    ax_x, ax_y = bg_lng.max() - 0.4, bg_lat.min() + 1.0
    ax.annotate(
        "N", xy=(ax_x, ax_y + 0.55), xytext=(ax_x, ax_y),
        ha="center", fontsize=13, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", lw=1.8, color="black"),
    )

## Plotting function

Draws the shared background, the park's similar cells (binned into 5 equal-interval classes
of `similarity_score`, colored with Turbo), and a per-park legend placed in the **top-left**
corner — the one part of the frame with no islands, so it never covers data (the original
bottom-left placement hid the Nansei/Ogasawara island chains). `add_scale_north=True` can be
passed to also draw the shared scale bar / north arrow onto a given figure.

In [ ]:
def plot_park(slug: str, cfg: dict, con: duckdb.DuckDBPyConnection, add_scale_north: bool = False) -> Path:
    thr, disp, typ = cfg["threshold"], cfg["display"], cfg["type"]
    gdf = get_park_similarity(slug, thr, con)
    lat, lng, score = gdf["lat"].values, gdf["lng"].values, gdf["similarity_score"].values

    mean_lat_rad = np.radians(bg_lat.mean())
    aspect = 1.0 / np.cos(mean_lat_rad)  # rough equirectangular correction for Japan's latitude

    fig, ax = plt.subplots(figsize=(7, 8.2))
    ax.scatter(bg_lng, bg_lat, s=0.4, c="#dcdcdc", linewidths=0, alpha=0.6, rasterized=True, zorder=1)

    vmax = score.max()
    edges = np.linspace(thr, vmax, 6)
    cmap = plt.get_cmap("turbo")
    colors5 = cmap(np.linspace(0.08, 0.95, 5))
    bin_idx = np.clip(np.digitize(score, edges[1:-1]), 0, 4)
    for b in range(5):
        m = bin_idx == b
        if m.sum() == 0:
            continue
        ax.scatter(lng[m], lat[m], s=1.6, color=colors5[b], linewidths=0, rasterized=True, zorder=2)

    ax.set_aspect(aspect)
    ax.set_xlim(bg_lng.min() - 0.3, bg_lng.max() + 0.3)
    ax.set_ylim(bg_lat.min() - 0.3, bg_lat.max() + 0.3)
    ax.set_title(f"{disp}\n{typ}  (n={len(lat):,} similar cells)", fontsize=20, pad=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # ── Legend: top-left corner (empty ocean; keeps southern island chains uncovered) ──
    box_x0, box_y0 = 0.02, 0.64
    box_w, box_h = 0.46, 0.34
    ax.add_patch(Rectangle((box_x0, box_y0), box_w, box_h, transform=ax.transAxes,
                            facecolor="white", edgecolor="0.3", linewidth=1.3, zorder=10))
    ax.text(box_x0 + 0.03, box_y0 + box_h - 0.045, "Similarity score",
            transform=ax.transAxes, fontsize=15, fontweight="bold", va="top", ha="left", zorder=11)
    ax.text(box_x0 + 0.03, box_y0 + box_h - 0.105, f"(\u2265 {thr:.3f})",
            transform=ax.transAxes, fontsize=14, fontweight="bold", va="top", ha="left", zorder=11)
    n_swatch = 5
    header_block = 0.155
    top_of_rows = box_y0 + box_h - header_block
    row_h = (box_h - header_block - 0.015) / n_swatch
    for b in range(n_swatch):
        y_center = top_of_rows - (b + 0.5) * row_h
        ax.add_patch(Rectangle((box_x0 + 0.03, y_center - row_h * 0.32), 0.07, row_h * 0.64,
                                transform=ax.transAxes, facecolor=colors5[b], edgecolor="none", zorder=11))
        ax.text(box_x0 + 0.13, y_center, f"{edges[b]:.3f} \u2013 {edges[b+1]:.3f}",
                transform=ax.transAxes, fontsize=14, va="center", ha="left", zorder=11)

    if add_scale_north:
        draw_scalebar_and_north_arrow(ax, bg_lat, bg_lng, km=200)

    num = disp.split(".")[0]
    name = disp.split(".", 1)[1].strip().replace("/", "-").replace("'", "")
    outpath = OUT_DIR / typ / f"{int(num):02d}_{name}.png"
    fig.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return outpath

## Generate all 35 maps

Reuses a single DuckDB connection across all 35 parks (row-group pruning makes each
extraction fast, ~1–3 s). Output PNGs are written to
`fix/paper_outputs/FigS1_check_new_threshold/{TypeI,TypeII,TypeIII,Island}/`.

In [ ]:
con = duckdb.connect()
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")

for slug, cfg in park_config.items():
    outpath = plot_park(slug, cfg, con)
    print(f"{outpath}")

## Shared north arrow / scale bar

All 35 maps use the identical extent, aspect correction and orientation, so the north
arrow and scale bar are added **once** here (re-rendering a single reference park) rather
than repeating identical elements on every tile. Place this single reference figure once
per composited Fig. S1 page in Illustrator; the other 34 tiles need no north arrow/scale
bar of their own.



If per-tile north arrows/scale bars are preferred instead, simply pass
`add_scale_north=True` in the loop above.

In [ ]:
reference_slug = "taisetsu"  # any park works; the scale bar/north arrow are identical for all

plot_park(reference_slug, park_config[reference_slug], con, add_scale_north=True)